# `02` RAG Knowledge — Grounding Answers in Institutional Sources
### Worksheet Step 2 · Layer 4 · ⏱ ~8 min
> **Setup:** standard library only — run cells top to bottom (`Shift`+`Enter`). See `00_Overview` for environment setup.

> **Worksheet prompt (L4):** *Which institutional sources will ground it?*

RAG (Retrieval-Augmented Generation) keeps the model honest: instead of
answering from memory, it answers from **your** documents — syllabi, policies,
repositories — and cites them. Here we build a tiny, dependency-free RAG over a
mock institutional knowledge base using pure-Python TF-IDF + cosine similarity.

In [1]:
import math, re
from collections import Counter

# --- mock institutional knowledge base (replace with YOUR sources) --------
KNOWLEDGE_BASE = {
    "policy_late_submission": (
        "Late assignment policy: submissions are accepted up to 7 days late "
        "with a 10% penalty per day. After 7 days a grade of zero is recorded "
        "unless a documented medical exemption is approved by the faculty."),
    "policy_academic_integrity": (
        "Academic integrity: unattributed AI-generated text is treated as "
        "plagiarism. Students must disclose AI assistance. Penalties range "
        "from resubmission to course failure."),
    "rubric_essay": (
        "Essay rubric (20 pts): thesis clarity 5, evidence and citations 5, "
        "structure 5, language and mechanics 5. A passing essay scores >= 10."),
    "research_ip_policy": (
        "Unpublished manuscripts and grant proposals are confidential IP of "
        "the university and authors. They must not be sent to external cloud "
        "AI services; only the on-premise system may process them."),
    "grading_appeals": (
        "Grade appeals must be filed within 14 days. A second faculty reviewer "
        "independently re-grades before any change is finalised."),
}

def tokenize(t):
    return re.findall(r"[a-z]+", t.lower())

def tfidf_index(docs):
    toks = {k: tokenize(v) for k, v in docs.items()}
    df = Counter()
    for ws in toks.values():
        df.update(set(ws))
    N = len(docs)
    idf = {w: math.log((N + 1) / (df[w] + 1)) + 1 for w in df}
    vecs = {}
    for k, ws in toks.items():
        tf = Counter(ws)
        vecs[k] = {w: (tf[w] / len(ws)) * idf[w] for w in tf}
    return vecs, idf

def cosine(a, b):
    common = set(a) & set(b)
    num = sum(a[w] * b[w] for w in common)
    da = math.sqrt(sum(v * v for v in a.values()))
    db = math.sqrt(sum(v * v for v in b.values()))
    return num / (da * db) if da and db else 0.0

DOC_VECS, IDF = tfidf_index(KNOWLEDGE_BASE)

def retrieve(query, k=2):
    q_tf = Counter(tokenize(query))
    n = sum(q_tf.values()) or 1
    q_vec = {w: (q_tf[w] / n) * IDF.get(w, 0.0) for w in q_tf}
    scored = sorted(((cosine(q_vec, DOC_VECS[d]), d) for d in KNOWLEDGE_BASE),
                    reverse=True)
    return [(d, round(s, 3)) for s, d in scored[:k] if s > 0]

print(retrieve("What happens if I submit my essay three days late?"))
print(retrieve("Can I upload an unpublished paper to ChatGPT?"))

[('policy_late_submission', 0.347), ('rubric_essay', 0.289)]
[('research_ip_policy', 0.221), ('policy_academic_integrity', 0.081)]


In [2]:
def rag_answer(query, k=2):
    hits = retrieve(query, k)
    context = "\n".join(f"[{d}] {KNOWLEDGE_BASE[d]}" for d, _ in hits)
    citations = [d for d, _ in hits]
    top_score = hits[0][1] if hits else 0.0
    return {"context": context, "citations": citations,
            "retrieval_score": top_score}

r = rag_answer("late essay penalty")
print("CITATIONS:", r["citations"])
print("TOP SCORE:", r["retrieval_score"])
print("CONTEXT:\n", r["context"])

CITATIONS: ['policy_late_submission', 'rubric_essay']
TOP SCORE: 0.296
CONTEXT:
 [policy_late_submission] Late assignment policy: submissions are accepted up to 7 days late with a 10% penalty per day. After 7 days a grade of zero is recorded unless a documented medical exemption is approved by the faculty.
[rubric_essay] Essay rubric (20 pts): thesis clarity 5, evidence and citations 5, structure 5, language and mechanics 5. A passing essay scores >= 10.


**Why this matters for the worksheet:** the *retrieval score* becomes an input
to your confidence calculation in `nb 03`. A query with no good match (score
near 0) should **not** get a confident answer — that's a hallucination waiting
to happen. Grounding + citations is your Layer-6 "Exemplary" criterion.

### ✏️ Your turn
1. Replace `KNOWLEDGE_BASE` with 2–3 real sources from your scenario.
2. Run a query that has **no** good source. What score do you get?
3. On the worksheet (L4), list the exact institutional sources you'd index.